# BloodMNIST Predictions
## Imports

In [1]:
from pathlib import Path
import numpy as np
import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from models.resnet18.models import ResNet18

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 32
output = Path('predictions')
output.mkdir(exist_ok=True)
print(device)

cuda


## Dataset

In [2]:
with np.load('data/bloodmnist.npz', allow_pickle=False) as data:
    splits = {split: (data[f'{split}_images'], data[f'{split}_labels'].ravel())
              for split in ('train', 'val', 'test')}

## Models

In [3]:
cnn = ResNet18(in_channels=3, num_classes=8)
checkpoint = torch.load('models/resnet18/resnet18_28_1.pth',
                        map_location='cpu', weights_only=True)
cnn.load_state_dict(checkpoint['net'])
vit = ViTForImageClassification.from_pretrained('models/vit', local_files_only=True)
processor = ViTImageProcessor.from_pretrained('models/vit', local_files_only=True)
models = {'resnet18': cnn.to(device).eval(), 'vit': vit.to(device).eval()}

## Predictions → CSV (image_index, label, prediction, p0–p7)

In [4]:
@torch.inference_mode()
def predict(name, model, images):
    probabilities = []
    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size]
        if name == 'resnet18':
            x = torch.from_numpy(batch.copy()).permute(0, 3, 1, 2).to(device).float()
            logits = model((x / 255 - 0.5) / 0.5)
        else:
            inputs = processor(images=list(batch), return_tensors='pt').to(device)
            logits = model(**inputs).logits
        probabilities.append(logits.softmax(dim=1).cpu().numpy())
    return np.concatenate(probabilities)

for name, model in models.items():
    for split, (images, labels) in splits.items():
        probabilities = predict(name, model, images)
        predictions = probabilities.argmax(axis=1)
        rows = np.column_stack((np.arange(len(labels)), labels, predictions, probabilities))
        np.savetxt(output / f'{name}_{split}.csv', rows, delimiter=',',
                   header='image_index,label,prediction,' + ','.join(f'p{i}' for i in range(8)),
                   comments='', fmt=['%d'] * 3 + ['%.9g'] * 8)
        print(f'{name} {split}: {len(labels)} predictions, accuracy {(predictions == labels).mean():.2%}')

resnet18 train: 11959 predictions, accuracy 100.00%


resnet18 val: 1712 predictions, accuracy 96.55%


resnet18 test: 3421 predictions, accuracy 95.79%


vit train: 11959 predictions, accuracy 99.48%


vit val: 1712 predictions, accuracy 97.78%


vit test: 3421 predictions, accuracy 96.99%
